# Ein erster Agent mit LangGraph

Ein kleines Rechenbeispiel macht den Ablauf sichtbar: **Modell → Werkzeug → Modell → Antwort**.

LangGraph steuert diesen Ablauf mit einem Graphen. Das Sprachmodell wählt Rechenwerkzeuge aus; Python führt die Berechnung aus.

**Am Ende:** ein selbst gebauter Agent, ein Diagramm des Ablaufs und ein Blick auf jede ausgeführte Station.


## Start

Die [README](../README.md) beschreibt Installation, API-Schlüssel und die lokale Umgebung.

Start im Repository-Ordner:

```bash
uv sync --locked
uv run python -m ipykernel install --sys-prefix --name langgraph-agent --display-name "LangGraph · langgraph-agent"
uv run jupyter lab notebooks/01_langgraph_quickstart.ipynb
```

Kernel: **LangGraph · langgraph-agent**.

Die Zellen sind für die Ausführung von oben nach unten angeordnet.

**API-Nutzung:** Nur die beiden Beispielzellen rufen Gemini auf. Dabei werden die eingegebenen Nachrichten an Google übertragen und es können Kosten entstehen.

Ein Neustart des Kernels setzt alle Python-Variablen zurück.


## 1 · Werkzeuge und Modell

Ein **Tool** ist hier eine Python-Funktion mit einer Beschreibung.

`@tool` macht Name, Parameter und Beschreibung für das Modell verfügbar. Die Beschreibung hilft bei der Werkzeugwahl.

Zuerst drei Funktionen, noch ohne Modellaufruf:


In [ ]:
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """Addiert die beiden ganzen Zahlen a und b."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multipliziert die beiden ganzen Zahlen a und b."""
    return a * b


@tool
def divide(a: int, b: int) -> float | str:
    """Dividiert die ganze Zahl a durch b. Division durch null ist nicht definiert."""
    if b == 0:
        return "Fehler: Division durch null ist nicht definiert. Kein Ergebnis erfinden."
    return a / b


tools = [add, multiply, divide]
tools_by_name = {current_tool.name: current_tool for current_tool in tools}


### Ein Werkzeug direkt testen

`invoke` führt das Werkzeug aus. Hier arbeitet ausschließlich Python.

Die Eingabe ist ein Dictionary, dessen Schlüssel zu den Funktionsparametern passen.


In [ ]:
print("Addition:", add.invoke({"a": 3, "b": 4}))
print("Division:", divide.invoke({"a": 8, "b": 2}))
print("Sonderfall:", divide.invoke({"a": 8, "b": 0}))


### Gemini anbinden

Der Schlüssel stammt aus `GOOGLE_API_KEY` oder `GEMINI_API_KEY`, bei Bedarf aus `.env` im Repository-Ordner.

Ohne hinterlegten Schlüssel erscheint eine verdeckte Eingabe. Der Schlüssel wird nicht ausgegeben und gehört nicht in das Notebook.

`bind_tools` stellt die Werkzeuge bereit. Es führt sie noch nicht aus.


In [ ]:
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI


repo_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "notebooks" / "01_langgraph_quickstart.ipynb").is_file()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Notebook aus dem Repository-Ordner oder notebooks/ starten.")

load_dotenv(repo_root / ".env")
key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
if not key:
    key = getpass("Google API-Schlüssel: ").strip()
if not key:
    raise ValueError("Für die Live-Beispiele fehlt ein Google API-Schlüssel.")

model = ChatGoogleGenerativeAI(
    model=os.getenv("GEMINI_MODEL", "gemini-3.8-flash"),
    api_key=key,
    vertexai=False,
)
model_with_tools = model.bind_tools(tools)
del key


## Drei Bausteine im Überblick

![Modell, Werkzeuge und gemeinsamer Zustand als Bausteine eines Agenten](../assets/agenten-bausteine.png)

*KI-generierte didaktische Illustration.* Das Modell wählt, die Werkzeuge rechnen, der Zustand sammelt die Nachrichten. Der ausführbare Graph folgt in Schritt 6.


## 2 · Den Zustand definieren

Der **State** ist das gemeinsame Arbeitsprotokoll eines Durchlaufs.

`messages` enthält Anfrage, Modellnachrichten und Werkzeugergebnisse.

`llm_calls` zählt die Modellaufrufe.

**Reducer:** `operator.add` hängt neue Nachrichten an die bestehende Liste an. Der Zähler wird dagegen durch seinen neuen Wert ersetzt.


In [ ]:
import operator
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage, ToolMessage


class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int


## 3 · Den Modellknoten definieren

Ein **Knoten** ist eine Station im Graphen. Hier bekommt das Modell die bisherigen Nachrichten und eine Arbeitsanweisung.

Die Rückgabe enthält nur die Änderungen am State: eine neue Modellnachricht und den erhöhten Zähler.

Die Definition ruft noch kein Modell auf.


In [ ]:
def llm_call(state: MessagesState) -> dict:
    """Erzeugt den nächsten Werkzeugaufruf oder die abschließende Antwort."""
    response = model_with_tools.invoke(
        [
            SystemMessage(
                content=(
                    "Du bist ein Rechenassistent. Antworte kurz auf Deutsch. "
                    "Verwende die bereitgestellten Werkzeuge für jede Berechnung. "
                    "Bei abhängigen Rechenschritten: erst das Werkzeugergebnis abwarten, "
                    "danach den nächsten Schritt ausführen. "
                    "Werkzeugfehler erklären und kein Ergebnis erfinden."
                )
            )
        ]
        + state["messages"]
    )
    return {
        "messages": [response],
        "llm_calls": state.get("llm_calls", 0) + 1,
    }


## Vom Wunsch zum Werkzeugergebnis

![Ein Modell fordert ein Werkzeug an; Python führt es aus und liefert eine verknüpfte Ergebnisnachricht](../assets/tool-aufruf.png)

*KI-generierte didaktische Illustration.* Ein Tool-Aufruf ist zunächst eine strukturierte Anforderung. Erst der nächste Knoten führt Python-Code aus.


## 4 · Den Werkzeugknoten definieren

Eine Modellnachricht kann mehrere `tool_calls` enthalten.

Jeder Aufruf enthält **Name**, **Argumente** und eine **ID**. Der Name wählt die Funktion aus, die Argumente werden an sie übergeben.

`ToolMessage` liefert das Ergebnis zurück. `tool_call_id` verbindet es eindeutig mit dem ursprünglichen Aufruf.


In [ ]:
def tool_node(state: MessagesState) -> dict:
    """Führt angeforderte Werkzeuge aus und liefert ihre Ergebnisse zurück."""
    results = []
    for tool_call in state["messages"][-1].tool_calls:
        selected_tool = tools_by_name[tool_call["name"]]
        observation = selected_tool.invoke(tool_call["args"])
        results.append(
            ToolMessage(
                content=str(observation),
                tool_call_id=tool_call["id"],
            )
        )
    return {"messages": results}


## 5 · Weiterrechnen oder beenden?

Eine **Kante** verbindet zwei Stationen. Eine bedingte Kante entscheidet anhand des aktuellen State über den nächsten Schritt.

Tool-Aufruf vorhanden → `tool_node`.

Kein Tool-Aufruf → `END`.

Das Modell kann deshalb auch direkt eine Textantwort geben.


In [ ]:
from langgraph.graph import END, START, StateGraph


def should_continue(state: MessagesState) -> Literal["tool_node", "__end__"]:
    """Wählt nach dem Modellknoten die nächste Station."""
    if state["messages"][-1].tool_calls:
        return "tool_node"
    return END


## 6 · Den Graphen bauen

`StateGraph` verbindet State, Knoten und Kanten.

Nach jedem Werkzeugergebnis geht es zurück zum Modell. Es kann weiterrechnen oder antworten.

`compile()` erzeugt den ausführbaren Agenten. Es startet noch keinen Durchlauf.


In [ ]:
agent_builder = StateGraph(MessagesState)

agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)

agent_builder.add_edge(START, "llm_call")
agent_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    {"tool_node": "tool_node", END: END},
)
agent_builder.add_edge("tool_node", "llm_call")

agent = agent_builder.compile()


### Den tatsächlichen Graphen anzeigen

Dieses Diagramm wird aus den Knoten und Kanten des kompilierten Agenten erzeugt.

Graphviz rendert lokal. Dafür wird kein Modell und kein Diagramm-Webdienst aufgerufen.

Ohne das Systemprogramm `dot` erscheint stattdessen die Graphstruktur als Mermaid-Text. Die README beschreibt die Graphviz-Installation.


In [ ]:
import shutil

from graphviz import Digraph
from IPython.display import Code, SVG, display


graph_structure = agent.get_graph()
graph_diagram = Digraph("Rechenagent")
graph_diagram.attr(rankdir="TB", bgcolor="white")
graph_diagram.attr("node", fontname="Helvetica", style="rounded,filled", fillcolor="#eef3f8")
graph_diagram.attr("edge", fontname="Helvetica", color="#64748b")

node_labels = {
    START: "Start",
    "llm_call": "Modell: llm_call",
    "tool_node": "Werkzeuge: tool_node",
    END: "Ende",
}
for node_id in graph_structure.nodes:
    graph_diagram.node(
        node_id,
        label=node_labels.get(node_id, node_id),
        shape="ellipse" if node_id in (START, END) else "box",
    )

for edge in graph_structure.edges:
    label = ""
    if edge.conditional:
        label = "Tool-Aufruf" if edge.target == "tool_node" else "Antwort fertig"
    graph_diagram.edge(
        edge.source,
        edge.target,
        label=label,
        style="dashed" if edge.conditional else "solid",
    )

if shutil.which("dot"):
    display(SVG(graph_diagram.pipe(format="svg")))
else:
    print("Graphviz dot fehlt. Der folgende Mermaid-Text beschreibt denselben Graphen:")
    display(Code(graph_structure.draw_mermaid(), language="text"))


## Beispiel A · 3 + 4

`invoke` führt den gesamten Graphen aus und liefert den fertigen State.

Erwartetes Rechenergebnis: **7**.

Die Ausgabe zeigt alle Nachrichten: Anfrage, Tool-Anforderung, Werkzeugergebnis und abschließende Antwort. Die genaue Formulierung kann variieren.


In [ ]:
result = agent.invoke(
    {
        "messages": [HumanMessage(content="Addiere 3 und 4 mit dem passenden Werkzeug.")],
        "llm_calls": 0,
    },
    config={"recursion_limit": 12},
)

for message in result["messages"]:
    message.pretty_print()

print("\nModellaufrufe:", result["llm_calls"])


### Die Schleife lesen

`HumanMessage` enthält die Anfrage.

`AIMessage` enthält eine Tool-Anforderung oder die Antwort des Modells.

`ToolMessage` enthält ein Werkzeugergebnis.

Nach dem Ergebnis prüft das Modell den nächsten Schritt. Deshalb benötigt auch eine einfache Rechnung üblicherweise zwei Modellaufrufe.


## Beispiel B · (3 + 4) × 5

Hier hängt der zweite Rechenschritt vom ersten ab. Erwartetes Ergebnis: **35**.

`stream(..., stream_mode="updates")` zeigt die State-Änderungen nach jedem Knoten. Der Graph wird dabei einmal ausgeführt.

Jedes Beispiel startet mit einer neuen Nachrichtenliste. Ohne Checkpointer gibt es hier kein Gesprächsgedächtnis zwischen den Durchläufen.

**Grenze:** `recursion_limit=12` begrenzt Graph-Schritte, nicht nur Modellaufrufe. Auch Werkzeugknoten zählen. Eine nicht beendete Schleife löst beim Erreichen der Grenze einen Fehler aus.


In [ ]:
for update in agent.stream(
    {
        "messages": [
            HumanMessage(
                content=(
                    "Berechne (3 + 4) mal 5. "
                    "Verwende zuerst add, danach multiply mit dem Ergebnis der Addition."
                )
            )
        ],
        "llm_calls": 0,
    },
    config={"recursion_limit": 12},
    stream_mode="updates",
):
    for node_name, changes in update.items():
        print(f"\n=== {node_name} ===")
        for message in changes.get("messages", []):
            message.pretty_print()
        if "llm_calls" in changes:
            print("Modellaufrufe bisher:", changes["llm_calls"])


## Kleine Experimente

**1 · Ein neues Werkzeug**

Eine Funktion `subtract(a, b)` mit `@tool` ergänzen.

Danach `tools`, `tools_by_name` und `model_with_tools` aktualisieren. Die Rechnung `9 - 4` als neue Anfrage starten.

**2 · Der direkte Weg**

Eine Anfrage ohne Rechnung verwenden, etwa „Was macht ein Rechenassistent?“.

Prüfen, ob das Modell direkt antwortet und der Werkzeugknoten übersprungen wird. Die Auswahl bleibt eine Modellentscheidung.

**3 · Das Arbeitsprotokoll**

Gedankenexperiment: `operator.add` aus dem State entfernen.

Welche Nachrichten würden dann nach jedem Knoten noch im State stehen?


## Lernkontrolle

**Wer berechnet die Summe: das Modell oder Python?**

<details>
<summary>Lösung</summary>

Die Funktion `add` berechnet die Summe in Python. Das Modell wählt das Werkzeug und seine Argumente aus und formuliert anschließend die Antwort.

</details>

**Warum führt ein Tool-Ergebnis zurück zum Modell?**

<details>
<summary>Lösung</summary>

Das Modell verarbeitet das Ergebnis. Es kann einen weiteren Tool-Aufruf anfordern oder eine abschließende Antwort erzeugen.

</details>

**Was geht ohne den Messages-Reducer verloren?**

<details>
<summary>Lösung</summary>

Neue Nachrichten ersetzen die bisherige Liste. Anfrage und vorherige Schritte fehlen dann im Arbeitsprotokoll. Auch der Kontext für weitere Modellaufrufe wäre unvollständig.

</details>

**Was verknüpft Tool-Aufruf und Ergebnis?**

<details>
<summary>Lösung</summary>

Die ID des Tool-Aufrufs. Sie wird in `ToolMessage.tool_call_id` übernommen.

</details>


## Grundlage und Anpassungen

Grundlage: [LangGraph Quickstart, Graph API und vollständiges Codebeispiel](https://docs.langchain.com/oss/python/langgraph/quickstart#full-code-example).

Die sechs Schritte folgen diesem Beispiel.

Anpassungen für die Lehre: Gemini statt Claude, deutsche Erläuterungen und Arbeitsanweisung, sichere Schlüsseleingabe, Behandlung der Division durch null, lokale Graphviz-Darstellung und ein zusätzlicher Streaming-Durchlauf.

Die beiden KI-generierten Bilder illustrieren die Begriffe. Das Graphviz-Diagramm zeigt dagegen die tatsächlich definierte Graphstruktur.
